# Introduction

## What is this simulation:
This notbook contains a simple demonstration for the *CityRide Micro-Mobility System*. <br>
It demonstrates how object orientated programming and design patterns can be used to build an extensible, modular and easy to maintain system.

The purpose is not to build a fully working product, but demonstrating OO design:
- Dynamic bahviour changes at runtime.
- Loose coupling between classes.
- Seperation of responsibilities.
- Expansion through new pricing plans, vehicles and city configurations.

The target audience is the CityRide development team, with somewhat limited OO experience, so the examples are kept small and focused.

## The notebook is organised into sections that define:
- The core domain classes (Vehicles, riders, zones and rides).
- Pricing and add-on behaviour.
- Fleet management.
- Cityfactory for city/area expansion.
- Observer listeners for billing and logging.

At the end of the notebook, a set of small demonstration examples are added, each demonstrating how these pinceples work together to make <br> the system modular, extensible and easier to maintain.

### How to run it
Run all cells in order from top to bottom. <br>
No external libraries are required. 

In [252]:
from abc import ABC, abstractmethod
from typing import Optional, List, Dict


#IDGenerator:
# A simple utility class responsible for generating unique IDs for Riders and Vehicles.
# By using a class level counter, it ensures that each ID is unique within the system.

# The reset() method is only used for testing purposes to reset the counters.


class IDGenerator:
    """Generates unique IDs for Riders and Vehicle inside the CityRide System."""
    vehicle_counter = 0
    rider_counter = 0

    @classmethod
    def new_vehicle_id(cls) -> str:
        """Returns a new unique vehicle ID."""
        cls.vehicle_counter += 1
        return f"Vehicle-{cls.vehicle_counter}"
    
    @classmethod
    def new_rider_id(cls) -> str:
        """Returns a new unique rider ID."""
        cls.rider_counter += 1
        return f"Rider-{cls.rider_counter}"
    
    @classmethod
    def reset(cls) -> None:
        """Reset counters (for testing purposes)."""
        cls.vehicle_counter = 0
        cls.rider_counter = 0

In [253]:
class Vehicle(ABC):
    """Base class for all CityRide vehicles.
    
    Encapulates shared state and behaviour:
    - Uniquw vehicle ID
    - Battery level
    - Current location
    - Whether the vehicle is in use or not
    - Whether the vehicle is in service or not
    
    Concrete vehicle types like: Scooter, bikes etc should inherit from this class 
    and implement their own start/end rules and battery usage."""

    # Minimum battery percentage to start a ride
    min_battery_to_start = 15.0

    def __init__(self, vehicle_id: str, battery_level: float, location: str):
        # Shared attributes for all vehicles in the system
        self.vehicle_id = vehicle_id 
        self.battery_level = battery_level # 0.0 to 100.0.
        self.location = location # Current location of the vehicle in the city.
        self.in_use = False # True when the vehicle is being used.
        self.in_service = True # Can be set to False if the vehicle needs maintenance.

    @abstractmethod
    def start_ride(self) -> None:
        """Mark the vehicle as in use and enforces rules that must be 
        checked before starting a ride (battery, location,  service status, etc)."""
        pass

    @abstractmethod
    def end_ride(self, new_location: str) -> None:
        """Mark the vehicle as free again and update its location
        when a ride has ended."""
        pass

    def drain_battery(self, minutes: float) -> None:
        """Default battery drain implementation. 
        
        Can be overridden by subclasses if they have 
        different battery usage."""
        drain_amount = minutes * 0.5  # Default drain rate: 0.5% per minute
        self.battery_level = max(0.0, self.battery_level - drain_amount)

        if self.battery_level <= self.min_battery_to_start:
            self.in_service = False
            print(f"Vehicle {self.vehicle_id} error: battery low.")

    def status(self) -> str:
        """Returns the current status summary of the vehicle's current state."""
        return (
            f"{self.__class__.__name__}"
            f"\n [ID={self.vehicle_id},"
            f"\n battery={self.battery_level:.1f}%,"
            f"\n location={self.location},"
            f"\n in_use={self.in_use},"
            f"\n in_service={self.in_service}]" 
        )
    
class Scooter(Vehicle):
    """Electric scooter with faster battery drain"""

    def __init__(self, battery_level: float, location: str):
        # Vehicle ID is auto-generated for each Scooter via the IDGenerator
        super().__init__(
            vehicle_id=IDGenerator.new_vehicle_id(),
            battery_level=battery_level,
            location=location
        )


    def start_ride(self) -> None:
        """Applies Scooter specific rules to start a ride."""
        if not self.in_service:
            print(f"[VEHICLE] Scooter {self.vehicle_id} is out of service.")
            return
        if self.battery_level < self.min_battery_to_start:
            print(f"[VEHICLE] Scooter {self.vehicle_id} has not enough battery to start a ride.")
            return
        if self.in_use:
            print(f"[VEHICLE] Scooter {self.vehicle_id} is already in use.")
            return
        self.in_use = True
        print(f"[VEHICLE] Scooter {self.vehicle_id} ride has started.")

    def end_ride(self, new_location:str) -> None:
        """Finish the ride and update the scooter's location."""
        if not self.in_use:
            print(f"[VEHICLE] Scooter {self.vehicle_id} is not currently in use.")
            return
        self.in_use = False
        self.location = new_location
        print(f"[VEHICLE] Scooter {self.vehicle_id} ride has ended. New location: {self.location}")

    def drain_battery(self, minutes: float) -> None:
        """Scooter drains battery faster than default vehicle."""
        drain_amount = minutes * 1.0  # Scooter drain rate: 1.0% per minute
        self.battery_level = max(0.0, self.battery_level - drain_amount)

        if self.battery_level <= self.min_battery_to_start:
            self.in_service = False
            print(f"[VEHICLE] Scooter {self.vehicle_id} error: battery low.")


class Bike(Vehicle):
    """Electric bike with slower battery drain."""

    def __init__(self, battery_level: float, location: str):
        # Vehicle ID is auto-generated for each Bike via the IDGenerator
        super().__init__(
            vehicle_id=IDGenerator.new_vehicle_id(),
            battery_level=battery_level,
            location=location
        )

    def start_ride(self) -> None:
        """Applies Bike specific rules to start a ride."""
        if not self.in_service:
            print(f"[VEHICLE] Bike {self.vehicle_id} is out of service.")
            return
        if self.battery_level < self.min_battery_to_start:
            print(f"[VEHICLE] Bike {self.vehicle_id} has not enough battery to start a ride.")
            return
        if self.in_use:
            print(f"[VEHICLE] Bike {self.vehicle_id} is already in use.")
            return
        self.in_use = True
        print(f"[VEHICLE] Bike {self.vehicle_id} ride has started.")

    def end_ride(self, new_location: str) -> None:
        """Finish the ride and update the bike's location."""
        if not self.in_use:
            print(f"[VEHICLE] Bike {self.vehicle_id} is not currently in use.")
            return
        self.in_use = False
        self.location = new_location
        print(f"[VEHICLE] Bike {self.vehicle_id} ride has ended. New location: {self.location}")

    def drain_battery(self, minutes: float) -> None:
        """Bike drains battery slower than default vehicle."""
        drain_amount = minutes * 0.3  # Bike drain rate: 0.3% per minute
        self.battery_level = max(0.0, self.battery_level - drain_amount)

        if self.battery_level <= self.min_battery_to_start:
            self.in_service = False
            print(f"[VEHICLE] Bike {self.vehicle_id} error: battery low.")

In [254]:
def apply_loyalty_discount(base_price: float, rider: "Rider") -> float:
    """
    Apply a simple loyalty discount based on loyalty points.

    Each loyalty point gives a 0.5% discount, up to a maximum of 25%  on the 
    base price. The discount amount is stored on the rider for later reference.
    """
    # Each loyalty point gives a 0.5% discount, up to a maximum of 25%
    discount_rate = min(rider.loyalty_points * 0.005, 0.25)
    discount_amount = base_price * discount_rate
    final_price = base_price - discount_amount
    
    # Stores the discount amount from this ride. This is not loyalty points,
    # but the actual discount used for billing/receipts or analytics.
    rider.last_discount_amount = discount_amount
    
    if discount_rate > 0:
        print(
            f"[PRICING] Loyalty discount ({discount_rate*100:.1f}%): "
            f"-{discount_amount:.2f} Kr. (from {base_price:.2f} kr.)"
        )

    return final_price

class PricingStrategy(ABC):
    """
    Interface for all rider pricing plans.
    Concrete pricing strategies encapsulates how the base price is
    calculated based on, startup fees, ride duration and the rider plan. 
    This keeps pricing rules flexible and allows plans to be swapped
    or modified with out changing the core code
    """

    @abstractmethod
    def calculate_price(self, rider: "Rider", minutes: float) -> float:
        """Return the total price for a ride, based on rider and ride duration."""
        pass

class PayAsYouGoPricing(PricingStrategy):
    """Simple Pay-As-You-Go pricing strategy, the default plan 
    (Higher price, no subscription)."""

    def __init__(self, startup_fee: float = 10.0, per_minute_fee: float = 3.0):
        self.startup_fee = startup_fee
        self.per_minute_fee = per_minute_fee
        
    def calculate_price(self, rider: "Rider", minutes: float) -> float:
        base = self.startup_fee + (self.per_minute_fee * minutes)
        return apply_loyalty_discount(base, rider)
        

class CommuterPricing(PricingStrategy):
    """Cheaper pricing for commuters with a monthly plan
    (Cheaper rates, needs subscription)."""
    def __init__(self, startup_fee: float = 5.0, per_minute_fee: float = 1.5):
        self.startup_fee = startup_fee
        self.per_minute_fee = per_minute_fee
        
    def calculate_price(self, rider: "Rider", minutes: float) -> float:
        base = self.startup_fee + (self.per_minute_fee * minutes)
        return apply_loyalty_discount(base, rider)
    
class TouristPricing(PricingStrategy):
    """Pricing for tourists on a short-term subscription.
    (Moderate rates, weekly subscription)"""
    def __init__(self, startup_fee: float = 7.5, per_minute_fee: float = 2.0):
        self.startup_fee = startup_fee
        self.per_minute_fee = per_minute_fee
        
    def calculate_price(self, rider: "Rider", minutes: float) -> float:
        base = self.startup_fee + (self.per_minute_fee * minutes)
        return apply_loyalty_discount(base, rider)
    
class AddOn(ABC):
    """Base class for ride add-ons( insurance, helmet, etc.)
    Each add-on can calculate an extra fee based on the base price
    and/or ride duration.
    """

    def __init__(self, name: str):
        self.name = name

    @abstractmethod
    def calculate_fee(self, base_price: float, minutes: float) -> float:
        "return the extra fee this add-on adds to the ride price."
        pass

class HelmetRental(AddOn):
    """Fixed-fee helmet rental add-on."""

    def __init__(self, fee_per_ride: float = 20.00):
        super().__init__(name="Helmet Rental")
        self.fee_per_ride = fee_per_ride

    def calculate_fee(self, base_price: float, minutes: float) -> float:
        return self.fee_per_ride
    
class Insurance(AddOn):
    """Insurance charged as a percentage of the base ride price."""

    def __init__(self, percentage_rate: float = 0.10):
        super().__init__("Ride insurance")
        self.percentage_rate = percentage_rate

    def calculate_fee(self, base_price: float, minutes: float) -> float:
        return base_price * self.percentage_rate
    

class AddOnPricing(PricingStrategy):
    """Decorator that wraps another PricingStrategy and adds add-on fees.
    
    This allows to stack multiple add-ons on top of any pricing plan
    without modifying the original strategy classes.
    """

    def __init__(self, base_strategy: PricingStrategy, add_ons: List[AddOn]):
        self.base_strategy = base_strategy
        self.add_ons = add_ons

    def calculate_price(self, rider: "Rider", minutes: float) -> float:
        # First, calculate the base price using the wrapped strategy.
        base_price = self.base_strategy.calculate_price(rider, minutes)

        # then accumulate extra fees from each add-on.
        extra_total = 0.0
        if self.add_ons:
            print("\nAdd-ons applied:")
        for add_on in self.add_ons:
            fee = add_on.calculate_fee(base_price, minutes)
            extra_total += fee
            print(f" - {add_on.name}: {fee:.2f} Kr.")
        
        total = base_price + extra_total
        return total

In [255]:
class Rider:
    """
    Rider Object that reperesent a user on the CityRide system.
    
    A rider has:
    - A unique ID.
    - A name.
    - A Pricing plan (Subscription: Pay-As-You-Go, Commuter, Tourist, etc)
    - The option for add-ons, like helemt rental, insurance. 
    (These can be wrapped around the pricing strategy via a decorator.)
    - Loyalty points that gives discounts on rides.
    
    """

    def __init__(self, name: str, plan: str, pricing_strategy: PricingStrategy):
        self.rider_id = IDGenerator.new_rider_id()
        self.name = name
        self.plan = plan

        # Base pricing strategy without any add-ons
        # This is needed so add-ons can be removed/restored without losing the original pricing plan.
        self.base_pricing_strategy = pricing_strategy

        # The active strategy used to calculate prices
        # This is the base strategy or a decorated version with add-ons.
        self.pricing_strategy = pricing_strategy

        # Loyalty points accumulated from past rides.
        # These are used to calculate discounts on future rides.
        self.loyalty_points = 0

        # List of active add-ons for this rider.
        self.add_ons: List[AddOn] = []

        # Stores the discount amount from the last ride.
        # This is not loyalty points, its the % discount used for billing/receipts or analytics.
        self.last_discount_amount: float = 0.0
    
    def add_loyalty_points(self, points: int) -> None:
        """Add loyalty points to the rider's account and log the new total."""
        self.loyalty_points += points
        print(f"[PRICING] {points} loyalty points added to {self.name}. Total points: {self.loyalty_points}")

    def calculate_ride_price(self, minutes: float) -> float:
        """Use the current pricing (potentially decorated with add-ons) 
        strategy to calculate the ride cost based on the duration."""
        return self.pricing_strategy.calculate_price(self, minutes)
    
    def subscribe_to_plan(self, plan_name: str, pricing_strategy: PricingStrategy) -> None:
        """
        Change which pricing plan the rider is subscribed to and update
        the underlying pricing strategy.
        
        If the rider already has add-ons, the decorated strategy will be 
        rebuilt on top of the new plan with the same add-ons.
        """
        self.plan = plan_name
        self.base_pricing_strategy = pricing_strategy

        if self.add_ons:
            self.pricing_strategy = AddOnPricing(pricing_strategy, self.add_ons)
        else:
            self.pricing_strategy = pricing_strategy
        print(f"[PRICING] {self.name} has subscribed to the {self.plan} plan.")

    def choose_add_ons(self, add_ons: List[AddOn]) -> None:
        """Attach or remove add-ons for this rider and update the active pricing strategy.
        
        When add-ons are added, we wrap the base pricing strategy in a
        decorator(AddOnPricing). If no add-ons are selected, we fall back to the base strategy.
        """
        self.add_ons = add_ons

        if self.add_ons:
            self.pricing_strategy = AddOnPricing(self.base_pricing_strategy, add_ons)
            names = ', '.join([add_on.name for add_on in add_ons])
            print(f"[PRICING] {self.name} has chosen the following add-ons: {names}.")
        else:
            self.pricing_strategy = self.base_pricing_strategy
            print(f"[PRICING] {self.name} has removed all add-ons.")



In [256]:
class Zone:
    """
    Represents a geographical zone in the CityRide system.
    
    Zones can:
    - Be normal ride zones, without any special rules.
    - Zones with extra fees for entering/leaving.
    - Illegal no-ride zones, where rides can not be parked/ended.
    """
    def __init__(self, name: str, no_ride: bool = False, zone_fee: float = 0.0):
        self.name = name
        self.no_ride = no_ride
        self.zone_fee = zone_fee
    
    def is_ride_allowed(self) -> bool:
        """Return True if rides are allowed in this zone."""
        return not self.no_ride
    
    def rules_summary(self) -> str:
        """Retun a description of the zone rules."""
        if self.no_ride:
            return f"Zone {self.name} is a no-ride zone."
        if self.zone_fee > 0.0:
            return f"{self.name}: extra zone fee {self.zone_fee:.2f} kr."
        return f"{self.name}: Normal zone, no extra fees."

class Bill:
    """Represents the result of one completed ride
    
    
    It stores:
    - Rider info (name, ID, plan).
    - Vehicle used (ID).
    - Start and end zones.
    - Duration and pricing breakdown (base price, zone fees, total).
    - Discount amount."""
    def __init__(self, rider_name: str, rider_id: str, plan: str, vehicle_id: str, start_zone: str, end_zone: str, minutes: float, base_price: float, zone_fee: float, total_price: float, discount_amount: float):
        self.rider_name = rider_name
        self.rider_id = rider_id
        self.plan = plan
        self.vehicle_id = vehicle_id
        self.start_zone = start_zone
        self.end_zone = end_zone
        self.minutes = minutes
        self.base_price = base_price
        self.zone_fee = zone_fee
        self.total_price = total_price
        self.discount_amount = discount_amount
    
    def summary(self) -> str:
        " Returns a readable invoice summary of the ride."
        return (
            f"Bill for {self.rider_name} (ID={self.rider_id})\n"
            f"Plan: {self.plan}\n"
            f"Vehicle ID: {self.vehicle_id}\n"
            f"From: {self.start_zone} To: {self.end_zone}\n"
            f"Duration: {self.minutes} minutes\n"
            f"Base Price: {self.base_price:.2f} kr.\n"
            f"Zone Fee: {self.zone_fee:.2f} kr.\n"
            f"Discount Amount: -{self.discount_amount:.2f} kr.\n"
            f"Total Price: {self.total_price:.2f} kr.\n"
        )

In [257]:
class RideEventListener(ABC):
    """
    Observer interface for objects that want to react when a ride has ended.
    
    Concrete listeners:
    - BillingSystem (Creates and stores bills).
    - RideLogger (Logs ride data for analytics/debugging).

    This keeps Ride loosely coupled to external systems.
    """

    @abstractmethod
    def on_ride_ended(self, ride: "Ride", bill: Bill | None) -> None:
        """Callback triggered by Ride when a ride has ended."""
        pass

class BillingSystem(RideEventListener):
    """
    Collects bills generated when rides end.
    
    This class demonstrates one concrete Observer that reacts to
    ride completion events by generating and storing invoices.
    """

    def __init__(self):
        self.bills: list[Bill] = []
    
    def on_ride_ended(self, ride: "Ride", bill: Bill | None) -> None:
        # If no bill was generated, like a ride that ended in a no-ride zone, it skips storing the bill.
        if bill is None:
            return
        self.bills.append(bill)
        print("BillingSystem: Bill stored.")
        print("\n[BILLING] --- Bill Summary ---")
        print(bill.summary())

class RideLogger(RideEventListener):
    """
    Logs ride information for analytics and debugging.
    
    Another concrete Observer that reacts to ride completion events.
    This demonstrates how multiple listeners can react independently 
    to the same ride event without interfering with each other or the 
    class Ride knowing about them."""

    def __init__(self):
        self.logs: list[str]= []
    
    def on_ride_ended(self, ride: "Ride", bill: Bill | None) -> None:

        status = "active" if ride.is_active else "ended"
        entry = (
            f"[LOG] Ride log - Rider: {ride.rider.name} (ID={ride.rider.rider_id}), "
            f"Vehicle: {ride.vehicle.vehicle_id}, "
            f"From: {ride.start_zone.name}, To: {ride.end_zone.name if ride.end_zone else 'Unknown'}, "
            f"Battery Level: ({ride.vehicle.battery_level:.1f}%), "
            f"Status: {status}"
        )
        self.logs.append(entry)
        print(entry)



In [258]:
class Ride:
    """
    Represents a ride session in the CityRide system, linking a Rider, Vehicle and Zones.
    
    A Ride:
    - Starts in a specific start zone.
    - Ends in a specific end zone.
    - Uses the rider's current pricing plan (and add-ons) to calculate the price.
    - Notifies registered listeners when the ride ends (for billing, logging).
    """

    def __init__(self, rider: Rider, vehicle: Vehicle, start_zone: Zone, is_active: bool = False):
        self.rider = rider
        self.vehicle = vehicle
        self.start_zone = start_zone
        self.end_zone: Optional[Zone] = None

        # Observers/listeners that listen for a ride to end (BillingSystem, RideLogger).
        self._listeners: list[RideEventListener] = []
        
        # Track whether the ride is currently active or not.
        self.is_active = is_active

    def add_listener(self, listener: RideEventListener) -> None:
        """Register a new listener to be notified when the ride ends."""
        self._listeners.append(listener)

    def _notify_listeners(self, bill: Bill | None) -> None:
        """
        Notify all registered listeners that the ride has ended.
        
        Listeners can decide how to react:
        - BillingSystem will store the bill (if any).
        - RideLogger will log the ride details for analytics/debugging.
        """
        for listener in self._listeners:
            listener.on_ride_ended(self, bill)

    def start(self) -> None:
        """
        Start the ride using the vehicle interface.
        
        This method:
        - Checks if the start zone allows rides.
        - Marks the ride as active.
        - Calls the vehicle's start_ride() method.
        """
        
        if not self.start_zone.is_ride_allowed():
            print(f"[RIDE] Cannot start ride in {self.start_zone.name}: this is a  NO-RIDE zone.")
            return
        self.is_active = True
        print(f"[RIDE] {self.rider.name} (ID={self.rider.rider_id}) is starting a ride in {self.start_zone.name} using {self.vehicle.__class__.__name__} {self.vehicle.vehicle_id}.")
        print(f"[VEHICLE] Starting battery level: {self.vehicle.battery_level:.1f}%")
        self.vehicle.start_ride()

    def end(self, end_zone: Zone, minutes: float | None = None) -> None:
        """
        End the ride and update the vehicle location and notifies Observers/listeners.
        
        If the end zone is a no-ride zone:
        - The ride remains active.
        - Obsercers are notified with bill=None.
        - The user is asked to park in an allowed zone somewhere else.
        """
        if not end_zone.is_ride_allowed():
            print(f"[RIDE] Cannot end ride in {end_zone.name}: this is a NO-RIDE zone.")
            print(f"[RIDE] Please park the vehicle in an allowed zone and try again.")
            print(f"[RIDE] RIDE REMAINS ACTIVE.")
            self.end_zone = end_zone
            self._notify_listeners(None)
            return
        self.end_zone = end_zone
        self.is_active = False
        print(f"[RIDE] {self.rider.name} (ID={self.rider.rider_id}) is ending the ride in {self.end_zone.name}." f" using {self.vehicle.__class__.__name__} {self.vehicle.vehicle_id}")
        self.vehicle.end_ride(new_location=self.end_zone.name)

        bill: Bill | None = None
        if minutes is not None:
            bill= self._bill_ride(minutes)
            self.vehicle.drain_battery(minutes)
            print(f"[VEHICLE] Vehicle: {self.vehicle.vehicle_id}, battery level after ride: {self.vehicle.battery_level:.1f}%")
        self._notify_listeners(bill)
    
    def _bill_ride(self, minutes: float) -> Bill:
        """
        Calculate and print total price for the ride, printes a short pricing breakdown,
        updates loyalty points and returns a bill object."""
        base_price = self.rider.calculate_ride_price(minutes)

        zone_fee = 0.0
        if self.end_zone is not None:
            zone_fee = self.end_zone.zone_fee
        
        total_price = base_price + zone_fee

        print(f"\n[RIDE] Ride duration: {minutes} minutes")
        print(f"[PRICING] Price for plan '{self.rider.plan}' (incl. add-ons): {base_price:.2f} kr.")
        if zone_fee > 0.0:
            print(f"[PRICING] Zone fee for ending in {self.end_zone.name}: {zone_fee:.2f} kr.")
        print(f"[PRICING] Total price for the ride: {total_price:.2f} kr.\n")

        # Adds loyalty points based on ride duration (1 point per minute)
        self.rider.add_loyalty_points(int(minutes))

        return Bill(
            rider_name=self.rider.name,
            rider_id=self.rider.rider_id,
            plan=self.rider.plan,
            vehicle_id=self.vehicle.vehicle_id,
            start_zone=self.start_zone.name,
            end_zone=self.end_zone.name if self.end_zone else "Unknown",
            minutes=minutes,
            base_price=base_price,
            zone_fee=zone_fee,
            discount_amount=self.rider.last_discount_amount,
            total_price=total_price
        )

In [259]:
class FleetManager:
    """Keeps track of all vehicles in the CityRide System."""

    def __init__(self):
        self._vehicles: dict[str, Vehicle] = {}
    
    def add_vehicle(self, vehicle: Vehicle) -> None:
        "Registers a new vehicle in the fleet."
        self._vehicles[vehicle.vehicle_id] = vehicle

    def get_vehicle(self, vehicle_id: str) -> Optional[Vehicle]:
        """Looks up a vehicle by its ID, returns None if not found."""
        return self._vehicles.get(vehicle_id)
    
    def show_vehicle_status(self, vehicle_id: str) -> None:
        """Prints the status of a vehicle if it exsist in the fleet."""
        vehicle=self.get_vehicle(vehicle_id)

        if vehicle is None:
            print(f"Vehicle: {vehicle_id} is not found in the fleet.")
        else:
            print(f"Vehicle: {vehicle_id} \n status:\n{vehicle.status()}")

    def all_vehicles(self) -> list[Vehicle]:
        """Return a list of all registered vehicles"""
        return list(self._vehicles.values())
    
    def available_vehicles(self) -> list[Vehicle]:
        """Return a list of vehicles are in service and not in use."""
        return [v for v in self._vehicles.values() if not v.in_use and v.in_service]

In [260]:
class CityFactory(ABC):
    """
    Abstract Factory for creating CityRide components.
    
    Each city can have its own factory that creates:
    - Zones with city-specific rules.
    - Pricing plans tailored to the city's market.
    - How the fleet is configured and populated with vehicles.
    """
    
    @abstractmethod
    def create_zones(self) -> Dict[str, Zone]:
        """Create and return all zones for this city with zone names as keys."""
        pass

    @abstractmethod
    def create_pricing_plans(self) -> Dict[str, PricingStrategy]:
        """Create and return all pricing plans for this city with plan names as keys."""
        pass

    @abstractmethod
    def create_fleetmanager(self, zones: Dict[str, Zone]) -> FleetManager:
        """
        Create a fleetmanager and populate it with vehicles.
        
        The zones parameter allows the factory to place vehicles in specific zones
        when initializing the fleet.
        """
        pass


    def create_scooter(self, battery_level: float, zone: Zone) -> Scooter:
        """Helper method to create a scooter in a specific zone."""
        return Scooter(battery_level=battery_level, location=zone.name)
    
    def create_bike(self, battery_level: float, zone: Zone) -> Bike:
        """Helper method to create a bike in a specific zone."""
        return Bike(battery_level=battery_level, location=zone.name)

class OsloCityFactory(CityFactory):
    """Factory for creating CityRide components specific to Oslo."""

    def create_zones(self) -> Dict[str, Zone]:
        # Define zones specific to Oslo with rules and fees.
        downtown = Zone("Downtown")
        campus = Zone("Campus")
        airport = Zone("Airport", zone_fee=15.0)
        museum = Zone("Museum District", no_ride=True)
        suburbs = Zone("Suburbs", zone_fee=5.0)
        return {
            "Downtown": downtown,
            "Campus": campus,
            "Airport": airport,
            "Museum District": museum,
            "Suburbs": suburbs
        }
    
    def create_pricing_plans(self) -> Dict[str, PricingStrategy]:
        # Oslo specific pricing plans
        return {
            "payg": PayAsYouGoPricing(),
            "commuter": CommuterPricing(),
            "tourist": TouristPricing()
        }
    
    def create_fleetmanager(self, zones: Dict[str, Zone]) -> FleetManager:
        """
        Create a FleetManager and populate it with Oslo-specific vehicles.
        
        Vehicles are placed in different zones as per Oslo's layout to simulate
        a realistic distribution.
        """
        fleet = FleetManager()

        # Add vehicles to fleet
        scooter1 = self.create_scooter(battery_level=90.0, zone=zones["Downtown"])
        bike1 = self.create_bike(battery_level=80.0, zone=zones["Campus"])
        scooter2 = self.create_scooter(battery_level=60.0, zone=zones["Suburbs"])

        fleet.add_vehicle(scooter1)
        fleet.add_vehicle(bike1)
        fleet.add_vehicle(scooter2)
        return fleet

# Design Question 1 - Which OO principle and design patterns are applied?
This simulation applies a small set of OO principles and design patterns to keep the system modular and easy to extend.

## OO Principles:

**Abstraction**

Core behaviours are defined through abstract base classes such as **Vehicle**, **PricingStrategy**, **AddOn** and **RideEventListener**.<br>
This lets the system plug in new vehicles, pricing plans, add-ons or listeners without changing the rest of the code.
<br>

**Encapsulation**

Each class exposes only the operations it's supposed to like: (start_ride(), calculate_ride_price(), end()),<br>
while the internal state (battery handling, pricing details, event wiring) stays inside the class.<br>
Other parts of the system dont need to know how these things work, only how to call them.
<br>

**Polymorphism**

Objects are used through their abstract types, not their concrete classes.<br>
Any **PricingStrategy** works with **Rider**, any **Vehicle** works with **Ride**, and any **RideEventListener** can react to ride events.<br> 
This is enables strategies, add-ons and vehicle types to be swapped seamlessly without affecting any other part of the system.
<br>

**Open/Closed Principle**

The system is ectendable without modifying exsisting logic.<br>
New cities, vehicles, pricing plans and add-ons are added by creating new subclasses, not by changing core classes.<br>
Demo 5 (BergenCityFactory + CargoBike + StudentPricing) is a direct example of this.
<br>

## Design patterns:

**Strategy Pattern - Pricing** 

All pricing plans (**PayAsYouGoPricing**, **CommuterPricing**, **TouristPricing** and **StudentPricing**) implement **PricingStrategy**.<br>
The rider simply switches strategy and the rest of the system continues unchanged.
<br>

**Decorator Pattern - Add-ons**

Add-ons like HelmetRental and Insurance wrap the base pricing strategy using **AddOnPricing**.<br>
This add extra pricing logic without modifyiing any pricing classes or the Rider itself.
<br>

**Observer Pattern - Ride Events**

**Ride** Notifies all registered listeners (**BillingSystem** and **RideLogger**) when a ride ends.<br>
This keeps billing and logging seperate from ride logic and lets multiple listeners respond independently, without **Ride** even knowing.
<br>

**Abstract Factory - City Setup**

**OsloCityFactory** and **BergenCityFactory** create zones, pricing plans and fleets for each city.<br>
This keeps the configuration out of the simulation logic and allows new cities to be added with their own specific configuration<br>
without rewriting the core code.



# Design Question 2 - How do these choices reduce coupling, improve extensibillity, and support maintainabillity?

The main goal of the design is to keep classes loosley connected, predictable and easy to extend without rewriting the system. <br>
Everything is structured so a new change affects as little code as possible.

### 1. Reducing coupling

**Programming to interfaces**

Most parts of the system depend on **abstract types**, not concrete classes:
- "Rider" depends on "PricingStrategy", not on a specific plan.
- "Ride" depends on "RideEventListener", not directly on "BillingSystem" or "RideLogger".
- "CityFactory" returns zones, plans and fleets without other classes knowing how they were created.

Example:
Class Ride:
    def add_listener(self, listener: RideEventListener) -> None:
        self._listeners.append(listeners)

Ride does not care which listener it receives, only that it follows the interface. <br>
That keeps the coupling low.


**Single Responsibillity**

Each class has one job:
- "Vehicle" hierarchy -> movement and battery state.
- "Rider" -> pricing plan, add-ons and loyalty points.
- "Ride" -> coordinates a single ride and triggers billing.
- "BillingSystem" and "RideLogger" -> react to ride completion.
- "FleetManager" -> lookup and manage vehicles.
- "CityFactory" -> configure zones, plans and the fleet.

Because responsibillities are seperated, classes dont need to know about each others internal details, which reduces <br>
dependencies and makes the system safer to change.



### 2. Improving Extensibility

**Strategy + Decorator**

Both patterns make behaviour plug-and-play:
- Adding a new plan only requires creating a new **PricingStrategy**.
- Adding a new optional feature only requires a new **AddOn**.

Example:
rider.subscribe_to_plan("student", StudentPricing())
rider.choose_add_ons([HelmetRental(), Insurance()])

No changes to **Ride**, **Rider**, **BillingSystem** or any exsisting plans.<br>
It only creates new classes, and no modifications to older ones.<br>
This is clean extensibility.


**Observer**

When adding a new reaction to a completed ride, like:
ride.add_listener(MyCustomAnalyticsListener())
**Ride** does not change. <br>
It just notifies the listener interface. <br>
This means that unlimited new behaviours can be added with zero risk to the exsisting code.



**Abstract Factory**

"OsloCityFactory" encapsulates how a city is configured with zones, pricing plans and vehicles. <br>
If CityRide launches in another city, a new factory like: "BergenCityFactory" can be added without modifying "Ride", "Rider" or the "FleetManager". <br>
Meaning that adding a new city, does not change the core code at all, it only creates a new factory class.



### 3. Supporting Maintainability

Because dependencies are small and well defined:
- Changes are localised, like changing the battery drain for bikes, will only affect bikes and not scooter, same with adjusting the price for each plans.<br>
- When it comes to bug fixing, since things are isolated, it is much easier to localize bugs and fix it.<br>
- Since the code is implemented in a clear and isolated patterns, the understanding and readability of the code gets easier for developers with limited understandig of OOP.<br>

Overal, the combination of these OOP principles and design patterns makes the system easy to extend, easy to fix and predictable to maintain. 


# Design Question 3 - How is the system prepared for future changes?

This design aims to make future changes as local and low risk as possible.<br>
Most expected changes can be handled by "Adding new classes" or changing the configuration of exsisting classes, instead of rewriting exsisting logic,<br> 
this makes it future-proof, especally when the project grows rapid. 

### 1. Future changes in pricing and business rules

if CityRide wants to change the pricing rules or adding new plans, like "student discounts", "night tariffs" or "day packages",<br> 
these can be added by creating new **PricingStrategy** like:
```python
Class StudentPricing(PricingStrategy):
    def __init__(self, startup_fee: float = 1.0, per_minute_fee: float = 1.0):
        self.startup_fee = startup_fee
        self.per_minute_fee = per_minute_fee
        
    def calculate_price(self, rider: "Rider", minutes: float) -> float:
        base = self.startup_fee + (self.per_minute_fee * minutes)
        return apply_loyalty_discount(base, rider)
```

Nothing insde "Ride", "Rider", "BillingSystem", and "FleetManager" needs to change.<br>
They all depend on the **PricingStrategy** interface, not the concrete classes.<br>
This makes the system resistant to changing business rules and expansion.



### 2. Future changes in vehicles and hardware

if a new vehicle type is introduced, like a CargoBike or E-Moped, they can be added as a subclass of "Vehicle" and<br> 
be defined to their own behaviour:
```python
class CargoBike(Vehicle):
    def start_ride(self):
    ....
    def end_ride(self, new_location):
    ....
    def drain_battery(self, minutes: float) -> None:
        drain_amount = minutes * 0.5  
        self.battery_level = max(0.0, self.battery_level - drain_amount)

        if self.battery_level <= self.min_battery_to_start:
            self.in_service = False
            print(f"CargoBike {self.vehicle_id} error: battery low.")
```
Everything else in the system continues to work because it interacts only with the **Vehicle abstraction**.
Ride handling, billing, zones and factories do not need to change.

### 3. Future changes in cities, zones and regulations

City specific rules are isolated inside the Abstract factory:
- "OsloCityFactory" currently configures zones, pricing plans and a fleet for Oslo.
- A Future **BergenCityFactory** or **TrondheimCityFactory** can provide new zone rules (new fees, new no-ride areas) without changing any ride, billing or vehicle code.

This makes it extremely easy to “copy” what works today and adjust each city's local rules without touching the simulation logic.



### 4. Future changes in analytics, logging and integrations

Because ride events use the **Observer pattern**, new listeners can be added with no modification to existing code:
```python
ride.add_listener(AnalyticsListener())
```
A new listener like  "AnalyticsListener", "MarketingListener" or "HotSpotDataListener" can be configured to whatever it needs:
```python
class AnalyticsListener(RideEventListener):
    def __init__(self):
        self.logs: list[str]= []
    
    def on_ride_ended(self, ride: "Ride", bill: Bill | None) -> None:
        entry = (
            f"Ride log - Rider: {ride.rider.name} (ID={ride.rider.rider_id}), "
            f"Vehicle: {ride.vehicle.vehicle_id}, "
            f"From: {ride.start_zone.name}, To: {ride.end_zone.name if ride.end_zone else 'Unknown'},"
            f"(battery after ride: {ride.vehicle.battery_level:.1f}%)"
        )
        self.logs.append(entry)
        print(entry)
        
```

Because **Ride** only depends on the **RideEventListener** interface, analytics and integrations can change independently without risking or changing the core logic.



### 5. Localised impact of change

Because responsibilities are separated across small focused classes, most changes affects only one or a few classes:
- Changing bike battery usage only affects **Bike** and not **Scooter** or billing.
- Adding a new pricing rule only affects the specific **PricingStrategy** class.
- Adding a new zone or fee is done entirely inside the **CityFactory**, only affecting that one city and not all the others.
- Adding new listeners does not affect **Ride** or other listeners.

This reduces the risk of breaking the system when new features are added and makes the long-term maintenance easier.

# Design Question 4 - Example code Snippets Demonstrating the Concepts

The following small snippets illustrates how the main Object Orientated Design principles and design patterns appear in the CityRide simulation.


### 1. Strategy Pattern - Pricing plans

```python
class CommuterPricing(PricingStrategy):
    def __init__(self, startup_fee: float = 5.0, per_minute_fee: float = 1.5):
        self.startup_fee = startup_fee
        self.per_minute_fee = per_minute_fee

    def calculate_price(self, rider: "Rider", minutes: float) -> float: 
        base = self.startup_fee + (self.per_minute_fee * minutes)
        return apply_loyalty_discount(base, rider)


# Usage
rider.subscribe_to_plan("commuter", CommuterPricing())
```
This demonstrates how pricing behaviour can be changed at runtime using interchangeable strategy classes.

### 2. Decorator Pattern - Add-ons

```python
class HelmetRental(AddOn):
    def __init__(self, fee_per_ride: float = 20.00):
        super().__init__(name="Helmet Rental")
        self.fee_per_ride = fee_per_ride

    def calculate_fee(self, base_price: float, minutes: float) -> float:
        return self.fee_per_ride

# Usage
rider.choose_add_ons([HelmetRental()])
```
This demonstrates how optional addons wrap the base pricing strategy and adds behaviour without modifying existing pricing classes.

### 3. Observer Pattern - Ride Event Listeners

```python
class BillingSystem(RideEventListener):
    def __init__(self):
        self.bills: list[Bill] = []
    
    def on_ride_ended(self, ride: "Ride", bill: Bill | None) -> None:
        if bill is None:
            return
        self.bills.append(bill)
        print("BillingSystem: Bill stored.")
        print(bill.summary())

# Usage
ride.add_listener(BillingSystem())
```
This demonstrates the observer pattern, the ride notifies listeners when a ride ends, allowing billing to run automatically without modifying the ride class.


### 4. Polymorphism - Diffrent listeners used interchangeably

```python
listeners = [BillingSystem(), RideLogger()]

for listener in listeners:
    listener.on_ride_ended(ride, bill)
```
BillingSystem and Ridelogger both implement the RideEventListener interface, so Ride can notify them polymorphically without knowing which concrete class it is calling.


### 5. Open/Closed Principle - Adding New Plans

```python
Class StudentPricing(PricingStrategy):
    def __init__(self, startup_fee: float = 1.0, per_minute_fee: float = 1.0):
        self.startup_fee = startup_fee
        self.per_minute_fee = per_minute_fee
        
    def calculate_price(self, rider: "Rider", minutes: float) -> float:
        base = self.startup_fee + (self.per_minute_fee * minutes)
        return apply_loyalty_discount(base, rider)

# Usage
rider.subscribe_to_plan("student", StudentPricing())
```

This demonstrates that new pricing behaviours can be added by extension without modifying existing classes.


### 6. Abstract Factory - Creating a City

```python
factory = OsloCityFactory()
zones = factory.create_zones()
fleet = factory.create_fleetmanager(zones)
plans = factory.create_pricing_plans()

# Usage
rider = Rider("Johan", "payg", plans["payg"])
vehicle = fleet.get_vehicle("Vehicle-1")
```
The Abstract factory groups all Oslo specific configuration in one place. <br>
The main code for Oslo does not need to know how zones, pricing plans or vehicles are created.


# Simulation Examples

This section contains small text-based simulations that demonstrate hot the CityRide design works in practice.

Each demo:
- Creates a small city configuration using "OsloCityFactory"
- Runs one or more rides
- Shows how the patterns discussed in DQ1-DQ4 appear in real usage.

### Demo 1 - Simple ride with Pay-As-You-Go Pricing

This Demo sets up Oslo using "OsloCityFactory", then creates a rider on the Pay-As-You-Go plan, and runs one ride from Downtown to Campus.
It illustrates:
- Basic use of "Ride", "Vehicle" and "Zone.
- Displaying the Strategy pattern in pricing.
- Displaying the Observer pattern through "BillingSystem" and "RideLogger".
- Displaying how battery drain and loyalty points works through composition and encapsulation.

In [261]:
# Resets the IDs so the output is predictable for the demo.
IDGenerator.reset()

# Setting up the city configuration for Oslo.
factory = OsloCityFactory()
zones = factory.create_zones()
fleet = factory.create_fleetmanager(zones)
plans = factory.create_pricing_plans()

# Choosing a vehicle from the fleet (the first available one).
vehicle = fleet.available_vehicles()[0]  

# Creating a rider on Pay-As-You-Go plan.
rider = Rider("Johan", "payg", plans["payg"])

# Setting up observers/listeners for billing and logging.
billing = BillingSystem()
logger = RideLogger()

# Create a ride session and register listeners.
ride = Ride(rider, vehicle, zones["Downtown"])
ride.add_listener(billing)
ride.add_listener(logger)

# Start and end the ride (10 minutes from Downtown to Campus).
ride.start()
ride.end(end_zone=zones["Campus"], minutes=10)

[RIDE] Johan (ID=Rider-1) is starting a ride in Downtown using Scooter Vehicle-1.
[VEHICLE] Starting battery level: 90.0%
[VEHICLE] Scooter Vehicle-1 ride has started.
[RIDE] Johan (ID=Rider-1) is ending the ride in Campus. using Scooter Vehicle-1
[VEHICLE] Scooter Vehicle-1 ride has ended. New location: Campus

[RIDE] Ride duration: 10 minutes
[PRICING] Price for plan 'payg' (incl. add-ons): 40.00 kr.
[PRICING] Total price for the ride: 40.00 kr.

[PRICING] 10 loyalty points added to Johan. Total points: 10
[VEHICLE] Vehicle: Vehicle-1, battery level after ride: 80.0%
BillingSystem: Bill stored.

[BILLING] --- Bill Summary ---
Bill for Johan (ID=Rider-1)
Plan: payg
Vehicle ID: Vehicle-1
From: Downtown To: Campus
Duration: 10 minutes
Base Price: 40.00 kr.
Zone Fee: 0.00 kr.
Discount Amount: -0.00 kr.
Total Price: 40.00 kr.

[LOG] Ride log - Rider: Johan (ID=Rider-1), Vehicle: Vehicle-1, From: Downtown, To: Campus, Battery Level: (80.0%), Status: ended


This output displays a complete ride executed using the Pay-As-You-Go plan.

- The ride starts in *Downtown* and ends at *Campus*.
- The vehicle is a scooter created via the "OsloCityFactory".
- Pricing is calculated using the Pay-As-You-Go pricing strategy.
- The Observer pattern notifies both the BillingSystem that stores the bill and the RideLogger that records a log entry without modifying the ride class.
- Battery drain is automatically applied according to the scooter's battery rules, in this case it is 1% per minute.
- Loyalty points are added to the rider account when the ride ends.

This demonstrates how the system's core components work together using composition, encapsulation and design patterns.

### Demo 2 - Switching from Pay-As-You-Go to Commuter

In this demo the same rider and vehicle are reused, but the rider switches from the Pay-As-You-Go plan to the Commuter plan before starting a new ride.<br>
This displays how the pricingstrategy can be swapped at "runtime" without chaning  the "Ride" or "Vehicle" classes.

In [262]:
# Rider swithes from Pay-As-You-Go to Commuter plan.
rider.subscribe_to_plan("commuter", plans["commuter"])

# New ride with the same scooter, but now using the commuter plan.
ride2 = Ride(rider, vehicle, zones["Campus"])
ride2.add_listener(billing)
ride2.add_listener(logger)

# start and end the second ride (10 minutes from Campus to Downtown).
ride2.start()
ride2.end(end_zone=zones["Downtown"], minutes=10)

[PRICING] Johan has subscribed to the commuter plan.
[RIDE] Johan (ID=Rider-1) is starting a ride in Campus using Scooter Vehicle-1.
[VEHICLE] Starting battery level: 80.0%
[VEHICLE] Scooter Vehicle-1 ride has started.
[RIDE] Johan (ID=Rider-1) is ending the ride in Downtown. using Scooter Vehicle-1
[VEHICLE] Scooter Vehicle-1 ride has ended. New location: Downtown
[PRICING] Loyalty discount (5.0%): -1.00 Kr. (from 20.00 kr.)

[RIDE] Ride duration: 10 minutes
[PRICING] Price for plan 'commuter' (incl. add-ons): 19.00 kr.
[PRICING] Total price for the ride: 19.00 kr.

[PRICING] 10 loyalty points added to Johan. Total points: 20
[VEHICLE] Vehicle: Vehicle-1, battery level after ride: 70.0%
BillingSystem: Bill stored.

[BILLING] --- Bill Summary ---
Bill for Johan (ID=Rider-1)
Plan: commuter
Vehicle ID: Vehicle-1
From: Campus To: Downtown
Duration: 10 minutes
Base Price: 19.00 kr.
Zone Fee: 0.00 kr.
Discount Amount: -1.00 kr.
Total Price: 19.00 kr.

[LOG] Ride log - Rider: Johan (ID=Rider

This output displays a second ride with the same rider and the same scooter, but using a diffrent pricing plan(Commuter).

- "subscribe_to_plan" changes the pricing strategy from Pay-As-You-Go to Commuter.
- The ride runs from *Campus* back to *Downtown*.
- The price is now calculated using the Commuter pricing plan, giving a diffrent cost for the same duration/distance, 19kr vs 40kr.
- Billing and logging still work unchanged via the Observer listeners.
- Since the rider have had previous rides and have some Loyalty points, the rider gets a discount based on his loyalty.

This demonstrates that pricing behaviour (Strategy pattern) can be changed at runtime by switching strategy objects (PricingStrategies), without modifying "Ride", "Vehivle" or any of the obeservers.

### Demo 3 - Add-ons with Decorator (Helmet + Insurance)

In this demo the rider keeps the *Commuter* plan, but adds Helmet and Insurance as add-ons. <br>
The "AddOnPricing" decerator wraps the base pricing strategy and adds extra fees on top of the normal price calculation.

In [263]:
# Rider chooses add-ons: Helmet and Insurance
rider.choose_add_ons([HelmetRental(), Insurance()])

# New ride with the same scooter, but now with add-ons.
ride3 = Ride(rider, vehicle, zones["Downtown"])
ride3.add_listener(billing)
ride3.add_listener(logger)

# start and end the ride, (15 minutes from Downtown to Airport).
# Airport has an extra zone fee.
ride3.start()
ride3.end(end_zone=zones["Airport"], minutes=15)

[PRICING] Johan has chosen the following add-ons: Helmet Rental, Ride insurance.
[RIDE] Johan (ID=Rider-1) is starting a ride in Downtown using Scooter Vehicle-1.
[VEHICLE] Starting battery level: 70.0%
[VEHICLE] Scooter Vehicle-1 ride has started.
[RIDE] Johan (ID=Rider-1) is ending the ride in Airport. using Scooter Vehicle-1
[VEHICLE] Scooter Vehicle-1 ride has ended. New location: Airport
[PRICING] Loyalty discount (10.0%): -2.75 Kr. (from 27.50 kr.)

Add-ons applied:
 - Helmet Rental: 20.00 Kr.
 - Ride insurance: 2.48 Kr.

[RIDE] Ride duration: 15 minutes
[PRICING] Price for plan 'commuter' (incl. add-ons): 47.23 kr.
[PRICING] Zone fee for ending in Airport: 15.00 kr.
[PRICING] Total price for the ride: 62.23 kr.

[PRICING] 15 loyalty points added to Johan. Total points: 35
[VEHICLE] Vehicle: Vehicle-1, battery level after ride: 55.0%
BillingSystem: Bill stored.

[BILLING] --- Bill Summary ---
Bill for Johan (ID=Rider-1)
Plan: commuter
Vehicle ID: Vehicle-1
From: Downtown To: Airp

This output displays a new ride using the same rider and scooter, but now with add-ons and a zone fee, creating a more complex pricing scenario.
- The rider selects extra services: Helmet Rental and Ride Insurance.
- The ride starts in *Downtown* and ends at the *Airport*, which adds a fee of 15kr to the price.
- Add-ons increase the base price before the discounts are applied.
- Since the rider has gotten more loyalty points, he get a bigger discount this time.
- Billing and logging still run automatically via the Observer listeners, without modifying any classes.

This demonstrates how add-ons, zone rules, loyalty discounts and observers all work together while keeping the system expandable and modular.

### Demo 4 - Ending a Ride in a No-Ride Zone

In this demo the rider starts a normal ride, but then tries to end it in the *Museum District*,  which is marked as a no-ride zone. <br>
This displays how zone restrictions are enforced without modifying the "Ride", "BillingSystem" or "RideLogger" classes.


In [264]:
# Reset IDs so the output is predictable for the demo
IDGenerator.reset()

# Setting up the city again
factory = OsloCityFactory()
zones = factory.create_zones()
fleet = factory.create_fleetmanager(zones)
plans = factory.create_pricing_plans()

# Create a rider on Pay-As-You-Go plan
rider = Rider("Johan", "payg", plans["payg"])

# Choosing a vehicle from the fleet
vehicle = fleet.available_vehicles()[0] 

# Setting up observers/listeners for billing and logging
billing = BillingSystem()
logger = RideLogger()

# Create a ride that starts in an allowed zone (Downtown)
ride4 = Ride(rider, vehicle, zones["Downtown"])
ride4.add_listener(billing)
ride4.add_listener(logger)

#start the ride
ride4.start()

# Try to end the rinde in a no-ride zone (Museum District)
ride4.end(end_zone=zones["Museum District"], minutes=10)

[RIDE] Johan (ID=Rider-1) is starting a ride in Downtown using Scooter Vehicle-1.
[VEHICLE] Starting battery level: 90.0%
[VEHICLE] Scooter Vehicle-1 ride has started.
[RIDE] Cannot end ride in Museum District: this is a NO-RIDE zone.
[RIDE] Please park the vehicle in an allowed zone and try again.
[RIDE] RIDE REMAINS ACTIVE.
[LOG] Ride log - Rider: Johan (ID=Rider-1), Vehicle: Vehicle-1, From: Downtown, To: Museum District, Battery Level: (90.0%), Status: active


This output displays what happens when a rider tries to end a ride insde a no-ride zone.<br>
The ride begins normally in an allowed zone (Downtown), but the system prevents the rider from ending the trip inside the *Museum District*, which is marked as No-ride.

It illustrates:
- Zone rules applied and enforced through the Zone class.
- Ride validation inside the Ride object.
- Correct use of the Observer pattern (BillingSystem + RideLogger), the bill is not created even though the rider tries to end, but the RideLogger still makes a log entry.
- In the log, you can see that the status of the scooter is still active.
- The battery stays at 90%, since the ride is still active. The system only updates the battery status once the ride successfully ends in a valid zone.
- How the system blocks invalid ride endings without modifying observers or pricing logic.

This demonstrates how zone rules can prevent invalid ride endings while keeping the system state consistent,  
without changing the "Ride", "BillingSystem" or "RideLogger" classes.


### Demo 5 - Adding a New City Configuration With a New Vehicle Type and Pricing Plan

This demo displays how the CityRide system can be extended with a new city (Bergen) that introduces:
- A new vehicle type (CargoBike).
- A new pricing plan (StudentPricing).
- A modified city configuration with its own fleet and zone rules.
- All without making any changes to the Ride, Vehicle, Pricing, Billing or Observer classes.

Using the Abstract Factory pattern, we implement "BergenCityFactory", which provides its own:
- Zones
- Fleet configuration
- Pricing plans

This demonstrates how the system supports new cities and behaviours by adding classes instead of modifying core logic, illustrating both Open/Closed principle and extensibility.

In [265]:
class CargoBike(Vehicle):
    """Heavier bike with slightly higher battery usage per minute."""

    def __init__(self, battery_level: float, location: str):
        # Vehicle ID is auto-generated for each vehicle via the IDGenerator
        super().__init__(
            vehicle_id=IDGenerator.new_vehicle_id(),
            battery_level=battery_level,
            location=location
        )

    def start_ride(self) -> None:
        """Applies CargoBike specific rules to start a ride."""
        if not self.in_service:
            print(f"[VEHICLE] CargoBike {self.vehicle_id} is out of service.")
            return
        if self.battery_level < self.min_battery_to_start:
            print(f"[VEHICLE] CargoBike {self.vehicle_id} has not enough battery to start a ride.")
            return
        if self.in_use:
            print(f"[VEHICLE] CargoBike {self.vehicle_id} is already in use.")
            return
        self.in_use = True
        print(f"[VEHICLE] CargoBike {self.vehicle_id} ride has started.")

    def end_ride(self, new_location: str) -> None:
        """Finish the ride and update the CargoBike's location."""
        if not self.in_use:
            print(f"[VEHICLE] CargoBike {self.vehicle_id} is not currently in use.")
            return
        self.in_use = False
        self.location = new_location
        print(f"[VEHICLE] CargoBike {self.vehicle_id} ride has ended. New location: {self.location}")

    def drain_battery(self, minutes: float) -> None:
        """CargoBike drains battery slightly faster than normal E-bike."""
        usage = 0.5 * minutes # CargoBike drain rate: 0.5% per minute
        self.battery_level = max(0.0, self.battery_level - usage)
        if self.battery_level <= self.min_battery_to_start:
            self.in_service = False
            print(f"[VEHICLE] CargoBike {self.vehicle_id} error: battery low.")

class StudentPricing(PricingStrategy):
    """Affordable pricing for students with valid ID.
    (Cheapest rates, requires student verification)"""

    def __init__(self, startup_fee: float = 1.0, per_minute_fee: float = 1.0):
        self.startup_fee = startup_fee
        self.per_minute_fee = per_minute_fee

    def calculate_price(self, rider: "Rider", minutes: float) -> float:
        base = self.startup_fee + (self.per_minute_fee * minutes)
        return apply_loyalty_discount(base, rider)
    
class BergenCityFactory(CityFactory):
    """Factory for creating CityRide components specific to Bergen."""

    def create_cargoBike(self, battery_level: float, zone: Zone) -> CargoBike:
        """Helper method to create a CargoBike in a specific zone."""
        return CargoBike(battery_level=battery_level, location=zone.name)

    def create_zones(self) -> Dict[str, Zone]:
        # Define zones specific to Bergen with rules and fees.
        downtown = Zone("Downtown")
        campus = Zone("Campus")
        airport = Zone("Airport", zone_fee=15.0)
        museum = Zone("Museum District", no_ride=True)
        suburbs = Zone("Suburbs", zone_fee=5.0)
        return {
            "Downtown": downtown,
            "Campus": campus,
            "Airport": airport,
            "Museum District": museum,
            "Suburbs": suburbs
        }
    
    def create_pricing_plans(self) -> Dict[str, PricingStrategy]:
        # Bergen specific pricing plans, added a Student plan.
        return {
            "payg": PayAsYouGoPricing(),
            "commuter": CommuterPricing(),
            "tourist": TouristPricing(),
            "student": StudentPricing()
        }
    
    def create_fleetmanager(self, zones: Dict[str, Zone]) -> FleetManager:
        """
        Create a FleetManager and populate it with Bergen-specific vehicles.
        
        In addtion to scooters and bikes, Bergen has a CargoBike in the fleet.
        """
        fleet= FleetManager()

        # Add standard vehicles to the Bergen fleet
        scooter1 = self.create_scooter(battery_level=90.0, zone=zones["Downtown"])
        bike1 = self.create_bike(battery_level=80.0, zone=zones["Campus"])
        scooter2 = self.create_scooter(battery_level=60.0, zone=zones["Suburbs"])

        fleet.add_vehicle(scooter1)
        fleet.add_vehicle(bike1)
        fleet.add_vehicle(scooter2)

        # Adding a CargoBike to the Bergen fleet
        cargo = self.create_cargoBike(battery_level=85.0, zone=zones["Campus"])
        fleet.add_vehicle(cargo)
        return fleet
    


#### About BergenCityFactory

This factory extends the system by providing a new city setup with its own combination of vehicles and pricing strategies.

key extensions:
- CargoBike is introduced as a new Vehicle with diffrent battery rules.
- StudentPricing is introduced as a new pricing strategy.
- The fleet now contains scooters, a bike and the new cargo bike, all of this added without modifying the original factory.
- All extensions plug into the existing framework through inheritance and polymorphism.

This factory replaces OsloCityFactory only for this demo, displaying how new cities can be added while keeping the ecisting ones intact and unchanged.

In [266]:
# Reset IDs so the output is predictable for the demo.
IDGenerator.reset()

# Set up the city using the extended Bergen factory.
factory = BergenCityFactory()
zones = factory.create_zones()
fleet = factory.create_fleetmanager(zones)
plans = factory.create_pricing_plans()

# Pick the CargoBike (it is the last vehicle added to the fleet).
vehicle = fleet.available_vehicles()[-1]

# Create a rider on the new Student plan.
rider = Rider("Tormod", "student", plans["student"])

# Set up observers/listeners for billing and logging.
billing = BillingSystem()
logger = RideLogger()

# Create a ride from Campus to Downtown using the CargoBike.
ride5 = Ride(rider, vehicle, zones["Campus"])
ride5.add_listener(billing)
ride5.add_listener(logger)

# Start and end the ride (12 minutes from Campus to Downtown).
ride5.start()
ride5.end(end_zone=zones["Downtown"], minutes=12)

[RIDE] Tormod (ID=Rider-1) is starting a ride in Campus using CargoBike Vehicle-4.
[VEHICLE] Starting battery level: 85.0%
[VEHICLE] CargoBike Vehicle-4 ride has started.
[RIDE] Tormod (ID=Rider-1) is ending the ride in Downtown. using CargoBike Vehicle-4
[VEHICLE] CargoBike Vehicle-4 ride has ended. New location: Downtown

[RIDE] Ride duration: 12 minutes
[PRICING] Price for plan 'student' (incl. add-ons): 13.00 kr.
[PRICING] Total price for the ride: 13.00 kr.

[PRICING] 12 loyalty points added to Tormod. Total points: 12
[VEHICLE] Vehicle: Vehicle-4, battery level after ride: 79.0%
BillingSystem: Bill stored.

[BILLING] --- Bill Summary ---
Bill for Tormod (ID=Rider-1)
Plan: student
Vehicle ID: Vehicle-4
From: Campus To: Downtown
Duration: 12 minutes
Base Price: 13.00 kr.
Zone Fee: 0.00 kr.
Discount Amount: -0.00 kr.
Total Price: 13.00 kr.

[LOG] Ride log - Rider: Tormod (ID=Rider-1), Vehicle: Vehicle-4, From: Campus, To: Downtown, Battery Level: (79.0%), Status: ended


This output displays a full ride using the BergenCityFactory, it demonstrates the ability to how the system evolves without modifying exsisting classes.

The output illustrates:
- The rider uses the Student pricing plan, which have a cheaper startup cost and minute price.
- The rider uses the new CargoBike vehicle, displaying diffrent battery consumption rules compared to scooters and bikes.
- City configuration was swapped simply by changing the factory, no changes were made in Ride, Billing, Logger or any other core class.
- Billing and logging work automatically via the Observer pattern, just like in previous demos.
- Loyalty points are still applied, displaying compatability between new features and exsisting functionality.

This demonstrates how the system supports future growth simply by adding new classes. <br>
Confirming extensibility, low coupling and adherence to OOP design principles.

## Final Conclusion

This simulation demonstrates how OOP principles and design patterns can create a system that is easy to extend and maintain. <br>
By separating most of the components, using interfaces and composition, the system supports new vehicles, zones, plans and pricing behaviour <br>
without modifying the core code. <br> The demonstrations highlights how Strategy, Factory, Decorator and Observer reduces coupling and prepare the design for future changes and requirements.<br>
<br>
Each demo targets a specific feature and shows how OOP can be used to achieve each goal without hard coding the core logic.<br>
Even though the simulation is simple, it still demonstrates how to avoid some of the same challenges found in systems in the real world. <br>
<br>
The goal of the assignment was to show how to build a system that is easy to change and can evolve efficently over time. <br> 
As a bonus OOP makes the system easy to maintain by keeping the functionallity isolated,<br>
which improves bug and error handling, espesally as the project grows and the codebase becomes more complex.
